# wandb-config-into-args — ex1: overwrite dataclass args from a wandb sweep config

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `wandb-config-into-args`. Running the final beacon cell reports progress against the `Logging: wandb.config into args` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Logging: wandb.config into args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`wandb-config-into-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "wandb-config-into-args"
DD_SUBTOPIC = "Logging: wandb.config into args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `args = update_args(args, dict(wandb.config))` — quick refresher

**The sweep pattern.** A wandb sweep samples hyperparameters and passes them to your `train()` function via `wandb.config`. Your dataclass `args` carries the same hparams as fields. You need to overwrite the args fields with whatever the sweep just sampled — otherwise `train()` uses its hardcoded defaults and the sweep has no effect.

**The recipe** (ARENA 0_3_8):

```python
def train():
    args = WandbResNetFinetuningArgs()       # hardcoded defaults
    wandb.init(...)                          # sweep agent populates wandb.config
    args = update_args(args, dict(wandb.config))  # overwrite from sweep
    trainer = WandbResNetFinetuner(args)
    trainer.train()
```

**`update_args` is a one-liner.** Loop over the sampled dict; for each key, `setattr(args, key, value)` if the field exists. Skip unknown keys (defensive — sweeps sometimes include meta keys like `_wandb`).

**`dataclasses.replace` is the prettier alternative.** `return replace(args, **sampled)` returns a NEW args with overrides applied — immutable and clean — but errors on unknown keys, so the mutable `update_args` is more robust to wandb's metadata fields.

### Exercise 1 — overwrite dataclass args from a wandb sweep config

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `setattr(args, k, v)` over a `wandb.config`-style dict to overwrite hparams on a dataclass `args` instance, skipping keys the dataclass doesn't define.
> Keywords: wandb, sweep, config, dataclass, args
> ```

**KCs targeted:** `wandb-config-overwrite-args`, `wandb-config-ignore-unknown`

Implement `ex1_update_args_from_wandb(args, sampled_config)`. The canonical ARENA 0_3_8 sweep recipe:

1. `args` is a dataclass instance with hparam fields (`lr`, `batch_size`, `weight_decay_bool`, plus an unrelated field `wandb_project`).
2. `sampled_config` is a plain dict (what `dict(wandb.config)` returns inside `train()`): may contain hparams to OVERWRITE on args, may contain UNKNOWN keys (wandb metadata like `'_wandb'`) that you should SILENTLY SKIP.
3. For each `(k, v)` in `sampled_config.items()`:
   - If `args` has an attribute `k` (use `hasattr`), set `setattr(args, k, v)`.
   - Otherwise, skip (don't error).
4. Return the (mutated) `args`.

This is `update_args` from ARENA's sweep cells.

In [ ]:
def ex1_update_args_from_wandb(args, sampled_config):
    for k, v in sampled_config.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args


<details><summary>Solution</summary>

```python
def ex1_update_args_from_wandb(args, sampled_config):
    for k, v in sampled_config.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args
```

**Why `hasattr` not `k in args.__dataclass_fields__`.** Both work for dataclasses, but `hasattr` also handles plain classes and inherited fields without special-casing. It's the most permissive check — exactly what you want for a defensive config-overwrite helper.

**Why SKIP unknown keys instead of raising.** Wandb sweeps add metadata keys like `_wandb` and `_runtime` automatically. If you raise on unknown keys, the sweep crashes on every run. Skipping is the conservative default.

**Mutation vs `dataclasses.replace`.** ARENA's example uses mutation because `update_args` is called inside `train()` and the args object is local. `dataclasses.replace(args, **sampled)` returns a NEW args — cleaner, but errors on unknown keys. The mutable form is what ARENA picks for robustness.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()